# Feature Engineering & Train/Val/Test Split

This notebook consumes the labeled dataset produced by **Labelling Strategy** and:

1. Engineers additional features from raw statistics
2. Inspects feature correlations and per-class distributions
3. Selects the final feature set
4. Performs a **stratified 70/15/15 split** with verified zero group overlap
5. Saves train/validation/test parquet files and a feature schema JSON

## Inputs
- `thermalwatch_labeled.parquet` (output of Labelling Strategy notebook)

## Outputs
| File | Description |
|---|---|
| `data/ml/train.parquet` | Training split (70 %) |
| `data/ml/validation.parquet` | Validation split (15 %) |
| `data/ml/test.parquet` | Test split (15 %) |
| `data/ml/feature_schema.json` | Feature columns, target, exclusions |
| `data/ml/split_summary.json` | Split sizes and class distributions |

## 0. Configuration

In [ ]:
import os

INPUT_PARQUET = os.environ.get("LABELED_PARQUET", "thermalwatch_labeled.parquet")
OUTPUT_DIR = os.environ.get("ML_DATA_DIR", "data/ml")

print(f"Input  : {INPUT_PARQUET}")
print(f"Output : {OUTPUT_DIR}")

## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import json

df = pd.read_parquet(INPUT_PARQUET)
df.shape

In [ ]:
df.dtypes

In [ ]:
df.columns.tolist()

In [ ]:
df['label'].value_counts()

## 2. Feature Engineering

### 2a. Log-transform FRP features

FRP (Fire Radiative Power) is heavy-tailed. Log1p-transform reduces skew and stabilises variance.

In [ ]:
df['log_mean_frp'] = np.log1p(df['mean_frp'])
df['log_std_frp'] = np.log1p(df['std_frp'].fillna(0))

df[['mean_frp', 'log_mean_frp', 'std_frp', 'log_std_frp']].describe()

### 2b. Temporal features

- `first_seen_month` — calendar month of first detection (captures seasonal signal)
- `active_duration_days` — span between first and last detection

In [ ]:
df['first_seen_month'] = df['first_seen'].dt.month
df['active_duration_days'] = (df['last_seen'] - df['first_seen']).dt.days

df[['first_seen_month', 'active_duration_days']].describe()

## 3. Feature Correlation & Class Profiling

In [ ]:
numeric_cols = ['obs_count', 'mean_frp', 'log_mean_frp', 'std_frp', 'log_std_frp',
                 'months_active', 'monsoon_obs_count', 'frp_cv', 
                 'nearest_osm_distance_km', 'active_duration_days', 'first_seen_month']

df[numeric_cols].corr()

### Per-class mean feature values

Industrial/mining classes show clear separation:
- Much higher `obs_count`, `months_active`, `active_duration_days`
- Much lower `nearest_osm_distance_km`
- Lower `mean_frp` (industrial fires are lower intensity but persistent)

In [ ]:
df.groupby('label')[numeric_cols].mean()

## 4. Feature Selection

Final feature set — chosen to minimise redundancy and leakage:

| Feature | Rationale |
|---|---|
| `obs_count` | Strongest single persistence proxy |
| `log_mean_frp` | Log-transformed FRP (replaces `mean_frp`) |
| `log_std_frp` | Log-transformed FRP std (replaces `std_frp`) |
| `frp_cv` | FRP variability ratio |
| `months_active` | Months with detections — very strong separator |
| `nearest_osm_distance_km` | Proximity to industrial features |
| `active_duration_days` | Calendar span |
| `first_seen_month` | Seasonality signal |

**Excluded:**
- `latitude`, `longitude` — excluded to prevent geographic memorization (leakage risk)
- `monsoon_obs_count` — redundant with `months_active` / `obs_count`
- `mean_frp`, `std_frp` — replaced by log-transformed versions

In [ ]:
feature_cols = [
    'obs_count',
    'log_mean_frp',
    'log_std_frp',
    'frp_cv',
    'months_active',
    'nearest_osm_distance_km',
    'active_duration_days',
    'first_seen_month',
]

## 5. Train / Validation / Test Split

**Strategy:** stratified 70 / 15 / 15 split using `sklearn.train_test_split`.

Each row is one unique `group_id` (physical location), so a group-level uniqueness check is performed to guarantee **zero data leakage** across splits.

In [ ]:
# Sanity check: each group_id appears exactly once
assert df['group_id'].nunique() == len(df), "Duplicate group_ids found!"
print("group_id uniqueness verified.")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

print("Train:", train_df.shape, train_df['label'].value_counts().to_dict())
print("Val:", val_df.shape, val_df['label'].value_counts().to_dict())
print("Test:", test_df.shape, test_df['label'].value_counts().to_dict())

In [ ]:
assert len(set(train_df['group_id']) & set(val_df['group_id'])) == 0
assert len(set(train_df['group_id']) & set(test_df['group_id'])) == 0
assert len(set(val_df['group_id']) & set(test_df['group_id'])) == 0
print("Verified: zero group overlap across train/val/test.")

## 6. Save Splits & Metadata

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_df.to_parquet(f"{OUTPUT_DIR}/train.parquet", engine="pyarrow", index=False)
val_df.to_parquet(f"{OUTPUT_DIR}/validation.parquet", engine="pyarrow", index=False)
test_df.to_parquet(f"{OUTPUT_DIR}/test.parquet", engine="pyarrow", index=False)

print("Saved train/validation/test parquet files.")

In [ ]:
feature_schema = {
    "feature_columns": feature_cols,
    "target_column": "label",
    "label_classes": sorted(df['label'].unique().tolist()),
    "excluded_features": {
        "latitude": "excluded to avoid geographic memorization (leakage risk)",
        "longitude": "excluded to avoid geographic memorization (leakage risk)",
        "monsoon_obs_count": "redundant with months_active/obs_count",
        "mean_frp": "replaced by log_mean_frp",
        "std_frp": "replaced by log_std_frp"
    },
    "notes": "raw lat/lon deliberately excluded from primary feature set"
}

with open(f"{OUTPUT_DIR}/feature_schema.json", "w") as f:
    json.dump(feature_schema, f, indent=2)

feature_schema

In [ ]:
split_summary = {
    "total_rows": int(len(df)),
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "split_method": "stratified 70/15/15 split by label; group-level uniqueness verified (one row = one physical source)",
    "train_class_distribution": train_df['label'].value_counts().to_dict(),
    "validation_class_distribution": val_df['label'].value_counts().to_dict(),
    "test_class_distribution": test_df['label'].value_counts().to_dict(),
    "random_state": 42
}

with open(f"{OUTPUT_DIR}/split_summary.json", "w") as f:
    json.dump(split_summary, f, indent=2)

split_summary